# Task 2: Unique Product Catalog using Semantic Embeddings

## Objective
The objective of this task is to identify duplicate or highly similar fashion products using semantic embeddings and create a clean product catalog by removing redundant entries.

## Dataset
The Fashion Product Images dataset contains product names and related metadata. Only the product names are used to generate semantic embeddings for duplicate detection.

## Methodology
1. Load the product catalog from the dataset.
2. Remove missing values and exact duplicate entries.
3. Generate semantic embeddings for product names using the SentenceTransformer (all-MiniLM-L6-v2) model.
4. Compute cosine similarity between product embeddings.
5. Group products with similarity greater than or equal to 0.85 as duplicate products.
6. Select one representative product from each duplicate group to create a unique product catalog.
7. Display duplicate clusters and the final cleaned catalog.

## Results
The system successfully identified semantically similar products and grouped them into duplicate clusters. From these groups, a unique product catalog was generated by keeping one representative product from each cluster.

## Conclusion
Semantic embeddings provide a more effective approach for duplicate detection than traditional keyword matching because they capture the contextual meaning of product names. This approach helps improve catalog quality by reducing redundancy while preserving unique products. Future improvements include clustering techniques such as FAISS or Agglomerative Clustering for handling larger datasets efficiently.

In [10]:
# Install required libraries
!pip install -q sentence-transformers scikit-learn

# Import libraries
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load dataset
df = pd.read_csv("styles.csv", on_bad_lines="skip")

# Keep only product names
df = df[['productDisplayName']].dropna()

# Remove exact duplicates
df = df.drop_duplicates()

# Use a larger sample for better duplicate detection
products = df['productDisplayName'].head(500).tolist()

print("Total Products Analysed:", len(products))

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings
embeddings = model.encode(products, show_progress_bar=True)

threshold = 0.85
visited = set()
duplicate_groups = []
unique_products = []

# Find duplicate products
for i in range(len(products)):

    if i in visited:
        continue

    group = [products[i]]
    visited.add(i)

    for j in range(i + 1, len(products)):

        if j in visited:
            continue

        similarity = cosine_similarity(
            [embeddings[i]],
            [embeddings[j]]
        )[0][0]

        if similarity >= threshold:

            group.append(products[j])
            visited.add(j)

    duplicate_groups.append(group)
    unique_products.append(group[0])

print("\n" + "="*60)
print("Duplicate Product Groups")
print("="*60)

count = 1

for group in duplicate_groups:

    if len(group) > 1:

        print(f"\nCluster {count}")

        print("-"*40)

        for item in group:
            print(item)

        count += 1

print("\n" + "="*60)
print("Unique Product Catalog")
print("="*60)

for product in unique_products:
    print(product)

print("\nSummary")
print("--------------------------")
print("Products Analysed :", len(products))
print("Duplicate Groups  :", count-1)
print("Unique Products   :", len(unique_products))

Total Products Analysed: 500


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]


Duplicate Product Groups

Cluster 1
----------------------------------------
Turtle Check Men Navy Blue Shirt
Turtle Check Men Yellow Shirt

Cluster 2
----------------------------------------
Peter England Men Party Blue Jeans
Peter England Men Party Black Jeans

Cluster 3
----------------------------------------
Titan Women Silver Watch
Titan Women White Watch

Cluster 4
----------------------------------------
Puma Men Grey T-shirt
Puma Men White and Navy Blue T-shirt

Cluster 5
----------------------------------------
Jealous 21 Women Purple Shirt
Jealous 21 Women Uaine Purple Tops

Cluster 6
----------------------------------------
Puma Men Pack of 3 Socks
Puma Women Pack of 3 Socks

Cluster 7
----------------------------------------
Fossil Women Black Huarache Weave Belt
Fossil Women Brown Huarache Weave Belt

Cluster 8
----------------------------------------
Murcia Women Blue Handbag
Murcia Women Casual Brown Handbag

Cluster 9
----------------------------------------
Police Me